# Bab 4: Data Cleaning (Pembersihan Data) - Penentu Kualitas Model AI/ML

## A. Definisi & Dampak pada Machine Learning
**Data Cleaning** adalah proses mendeteksi, memperbaiki, atau menghapus data yang rusak, salah format, duplikat, atau kosong (*missing values*) dari dataset. 

Di dalam industri AI/ML, tahapan ini adalah fase paling krusial. Model Machine Learning sangat sensitif terhadap kualitas data. Data yang kotor akan menghasilkan pola pembelajaran yang sesat, bias, dan tidak akurat saat diuji pada data riil di lapangan.

---

## B. 🚨 Alur Proses Eksekusi Wajib (Standard Operating Procedure)
Dalam membersihkan data untuk kebutuhan AI/ML, urutan eksekusi **TIDAK BOLEH ACARK**. Kamu wajib mengikuti urutan logis berikut agar tidak terjadi eror kalkulasi atau kehilangan informasi penting:

1. **Step 1: Eliminasi Duplikat (`.drop_duplicates()`)**
   * *Alasan:* Harus dilakukan paling pertama agar baris kembar tidak ikut merusak perhitungan nilai statistik (Mean/Median) pada langkah berikutnya.
2. **Step 2: Penanganan Data Kosong / Missing Values (`.fillna()` / `.dropna()`)**
   * *Alasan:* Mengisi atau membuang baris kosong secara objektif setelah data duplikat bersih.
3. **Step 3: Perbaikan Tipe Data (`.astype()`)**
   * *Alasan:* Memastikan semua kolom numerik benar-benar bertipe angka (`int64`/`float64`) dan kolom teks bertipe string/kategori, agar siap dibaca oleh algoritma ML.

---

## C. Eksekusi Teknik Pembersihan Data

### 1. Menangani Data Duplikat
Data duplikat dibuang agar model ML tidak mendewakan satu pola secara berlebihan (*overrepresentation*).
```python
import numpy as np
import pandas as pd

# Dataset Contoh awal yang kotor
data = {
    "Nama": ["Ali", "Budi", "Ali", "Dedi", "Evi"],
    "Usia": [23, 45, 23, np.nan, 29],
    "Tekanan_Darah": [120, 140, 120, 130, np.nan],
}
df = pd.DataFrame(data)

# Cek keberadaan duplikat (True/False)
print(df.duplicated())

# Eksekusi Wajib Langkah 1: Hapus duplikat dan timpa DataFrame asli
df = df.drop_duplicates()
```

### 2. Menangani Data Kosong (`NaN`) - Teknik Imputasi Statistik
**⚠️ Aturan Emas Industri:** Dilarang keras meng-hardcode nilai konstan numerik secara asal atau subjektif tanpa validasi *Domain Expert*. Gunakan pendekatan statistik objektif dari data itu sendiri:
* **Mean (Rata-rata):** Tepat digunakan jika sebaran data merata dan normal (tidak ada nilai ekstrem)
* **Median (Nilai Tengah):** Tepat digunakan untuk berjaga-jaga jika data memiliki pencilan (*outliners*) ekstrem agar nilai pengganti tidak ikut rusak

```python
# Kolom Usia diisi dengan Mean (Rata-rata)
rata_usia = df["Usia"].mean()
df["Usia"] = df["Usia"].fillna(rata_usia)

# Kolom Tekanan Darah diisi dengan Median (Nilai Tengah)
median_tensi = df["Tekanan_Darah"].median()
df["Tekanan_Darah"] = df["Tekanan_Darah"].fillna(median_tensi)
```

### 3. Menagani Data Kosong (`Nan`) - Teknik Penghapusan
Taktik menghapus baris hanya disarankan jika jumlah total baris dataset sangat besar (ratusan ribu) dan data yang bolong hanya sedikit (di bawah 5%). Jika data sedikit lalu langsung dihapus, model ML akan kekurangan bahan belajar

```python
# Opsi A: Hapus seluruh baris jika ada satu saja kolom yang NaN
df_bersih_total = df.dropna()

# Opsi B: Hanya hapus baris jika NaN berada di kolom spesifik yang krusial
df_bersih_spesifik = df.dropna(subset=["Usia"])
```

### 4. Perbaikan & Standarisasi Tipe Data
Efek samping dari pengisian nilai *Mean* (yang menghasilkan desimal) bisa merubah tipe data kolom bulat (*Integer*) menjadi *float*. Kita harus mengembalikannya ke tipe data yang benar

```python
# Mengubah tipe data Usia kembali menjadi integer murni
df["Usia"] = df["Usia"].astype("int64")

# Memastikan tipe data akhir untuk kesiapan ML
print(df.dtypes)
```

---

In [1]:
import pandas as pd
import numpy as np

In [2]:
# dataeset
data = {
    "Nama": ["Ali", "Budi", "Ali", "Dedi", "Evi"],
    "Usia": [23, 45, 23, np.nan, 29],
    "Tekanan_Darah": [120, 140, 120, 130, np.nan],
}
df = pd.DataFrame(data)

In [4]:
# Menangani Data Duplikat
# 1. Memeriksa apakah ada data yang duplikat
display(df.duplicated())

# 2. Menghapus data duplikat 
# Menimpa variabel dengan data yang sudah bersih
df = df.drop_duplicates()
display(df)

0    False
1    False
2     True
3    False
4    False
dtype: bool

,Nama,Usia,Tekanan_Darah
0,Ali,23.0,120.0
1,Budi,45.0,140.0
3,Dedi,NaN,130.0
4,Evi,29.0,NaN


In [ ]:
# Menangani Data Kosong
# Nilai pengganti dapat berisi:
# mean -> bagus jika data menyebar normal
# median -> bagus jika terdapat outliners yang ekstrem

# 1. Copy data asli
df_impute = df.copy()

# 2. Menghitung nilai rata-rata dan median dari kolom usia
mean_usia = df_impute["Usia"].mean()
median_tekanan_darah = df_impute["Tekanan_Darah"].median()

# 3. Mengisi NaN di kolom usia dengan rata-rata dan kolom tekanan darah dengan median
df_impute["Usia"] = df_impute["Usia"].fillna(mean_usia)
df_impute["Tekanan_Darah"] = df_impute["Tekanan_Darah"].fillna(median_tekanan_darah)

df_impute

,Nama,Usia,Tekanan_Darah
0,Ali,23.000000,120.0
1,Budi,45.000000,140.0
3,Dedi,32.333333,130.0
4,Evi,29.000000,130.0


In [10]:
# Mengubah Tipe Data Kolom
df_impute["Usia"] = df_impute["Usia"].astype("int64")
df_impute

,Nama,Usia,Tekanan_Darah
0,Ali,23,120.0
1,Budi,45,140.0
3,Dedi,32,130.0
4,Evi,29,130.0
